# RAG Arena: same query, 3 embedding providers, one LLM

The classic RAG pattern in ~150 lines: text corpus → chunk → embed → retrieve top-k → LLM answers with citations.

But the *interesting* version: **swap the embedding provider in one line and see how the retrieved chunks change**. Same query, three embedding spaces (OpenAI / Mistral / Cohere), three different sets of top-k chunks, three different LLM answers. That's where retrieval quality actually lives — and where Eden AI's swap-providers angle pays off most.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv

## 1. Configuration

Three embedding providers, all verified to work on the Eden AI gateway. One LLM (Claude) does the final answer — we control for the LLM so the only variable is *which chunks got retrieved*.

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_BASE = "https://api.edenai.run/v3"

EMBEDDING_PROVIDERS = [
    {"label": "OpenAI",  "model": "openai/text-embedding-3-small", "dim": 1536},
    {"label": "Mistral", "model": "mistral/mistral-embed",        "dim": 1024},
    {"label": "Cohere",  "model": "cohere/embed-english-v3.0",    "dim": 1024},
]

LLM_MODEL = "anthropic/claude-sonnet-4-5"  # answer model, fixed across panels
TOP_K = 3

# Toy corpus: 12 short paragraphs from a hypothetical company knowledge base
CORPUS = [
    "Cinder is an observability platform for ML pipelines. It surfaces data drift, model regressions, and prompt-injection attempts in real time. The product launched in March 2024.",
    "To install the Cinder SDK, run `pip install cinder-sdk`. Initialize it with your API key from the dashboard at app.cinder.dev/settings. The free tier includes 50k events per month.",
    "Cinder's pricing has three tiers. Free includes 50k events/month, the Team plan at $99/month adds team accounts and 1M events, and Enterprise starts at $2,000/month with SSO and on-prem options.",
    "The CTO of Cinder is Anita Verma, previously a staff engineer at Stripe. The CEO is Marc Bellot, ex-product lead at Datadog. They co-founded the company in 2023 in San Francisco.",
    "Data drift detection works by comparing the distribution of incoming features against a baseline window. The default window is 7 days. You can change this via `cinder.set_drift_window(days=30)`.",
    "Prompt injection detection uses a hybrid approach: a fine-tuned BERT classifier for known patterns plus an LLM-judge for novel attempts. False positive rate is below 0.3% on the public PINT benchmark.",
    "Cinder integrates with PyTorch, TensorFlow, scikit-learn, Hugging Face Transformers, and LangChain out of the box. There is also a generic REST hook for any custom framework.",
    "The dashboard offers four main views: Drift, Performance, Incidents, and Cost. The Cost view tracks token spend per model and per route, with daily/weekly/monthly breakdowns.",
    "Cinder is SOC 2 Type II certified and HIPAA-ready. Data is processed in the customer's region (US, EU, or APAC). On-prem deployment is available on the Enterprise plan.",
    "Our customer Bloom Bottle uses Cinder to monitor the demand-forecasting model that schedules production. They reduced stockouts by 28% in Q4 2024 after catching a feature-pipeline bug via drift alerts.",
    "For prompt-injection alerts, you can configure auto-mitigation actions: redact the input, block the request, or quarantine to a review queue. The default is alert-only.",
    "To upgrade from Team to Enterprise, contact sales@cinder.dev. The migration includes a dedicated onboarding engineer and an SLA review. Typical timeline is 2–3 weeks.",
]

SAMPLE_QUESTIONS = [
    "How much does Cinder cost?",
    "Who founded Cinder and where are they based?",
    "How do I detect prompt injection with Cinder?",
    "Is Cinder safe to use for healthcare data?",
    "Which frameworks does Cinder integrate with?",
    "What's an example of a customer success story?",
]


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Embeddings on sandbox may be mocked, which '
        'makes the retrieval comparison meaningless. Use a production key.</div>'
    ))

## 2. Embed + retrieve

For each embedding provider:
1. Embed every chunk in the corpus (one batched call)
2. Embed the query (one call)
3. Compute cosine similarity, return top-K chunks

The corpus is small (12 chunks) so we recompute on every query for simplicity. For a real corpus you'd cache embeddings or use a vector DB — the swap point is the same.

In [ ]:
import asyncio
import math
import time

import aiohttp

MAX_RETRIES = 2


async def _call_with_retry(session, url, payload):
    headers = {"Authorization": f"Bearer {EDENAI_API_KEY}", "Content-Type": "application/json"}
    for attempt in range(MAX_RETRIES + 1):
        async with session.post(url, headers=headers, json=payload,
                                timeout=aiohttp.ClientTimeout(total=90)) as resp:
            body = await resp.text()
            if resp.status == 200:
                return json.loads(body)
            if resp.status in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            raise RuntimeError(f"HTTP {resp.status}: {body[:200]}")
    raise RuntimeError("exhausted retries")


async def embed_batch(session, model_id, texts):
    """Embed a list of texts in a single call when the provider supports it,
    falling back to per-item calls if the batched request errors out."""
    try:
        data = await _call_with_retry(session, f"{EDENAI_BASE}/llm/embeddings",
                                       {"model": model_id, "input": texts})
        return [item["embedding"] for item in data["data"]]
    except Exception:
        # Fallback: one-at-a-time
        out = []
        for t in texts:
            data = await _call_with_retry(session, f"{EDENAI_BASE}/llm/embeddings",
                                           {"model": model_id, "input": t})
            out.append(data["data"][0]["embedding"])
        return out


def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb) if na and nb else 0.0


async def retrieve(session, provider, query, corpus, k):
    start = time.perf_counter()
    # Embed query + corpus together so they share a request
    embeds = await embed_batch(session, provider["model"], [query] + corpus)
    q_vec = embeds[0]
    chunk_vecs = embeds[1:]
    scored = [(i, cosine(q_vec, v)) for i, v in enumerate(chunk_vecs)]
    scored.sort(key=lambda x: -x[1])
    top = scored[:k]
    return {
        "provider": provider["label"],
        "model":    provider["model"],
        "dim":      provider["dim"],
        "latency":  time.perf_counter() - start,
        "top":      [{"chunk_idx": i, "score": s, "text": corpus[i]} for i, s in top],
    }


async def answer_with_chunks(session, llm_model, question, chunks):
    context = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))
    prompt = (
        "Answer the user's question using ONLY the numbered context below. "
        "Cite the source numbers in brackets like [1] or [2,3] after each claim. "
        "If the context doesn't contain the answer, say so plainly — do not invent.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}"
    )
    start = time.perf_counter()
    data = await _call_with_retry(session, f"{EDENAI_BASE}/llm/chat/completions",
                                   {"model": llm_model, "messages": [{"role": "user", "content": prompt}]})
    return {
        "answer":  data["choices"][0]["message"]["content"],
        "latency": time.perf_counter() - start,
    }

## 3. UI

In [ ]:
from ipywidgets import (
    Button, Dropdown, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Text, VBox,
)
from IPython.display import display

question_box = Text(
    value=SAMPLE_QUESTIONS[0],
    placeholder="Ask a question about the corpus…",
    layout=Layout(width="100%"),
)
sample_dropdown = Dropdown(
    options=[(q, q) for q in SAMPLE_QUESTIONS],
    value=SAMPLE_QUESTIONS[0],
    description="Sample:",
    layout=Layout(width="560px"),
)


def _on_sample_change(change):
    if change["name"] == "value":
        question_box.value = change["new"]


sample_dropdown.observe(_on_sample_change, names="value")

ask_btn = Button(description="🔍 Ask", button_style="primary")
clear_btn = Button(description="Clear")

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="420px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in EMBEDDING_PROVIDERS]
headers = [HTMLWidget() for _ in EMBEDDING_PROVIDERS]


def _empty_header(i):
    p = EMBEDDING_PROVIDERS[i]
    return (
        f'<div style="font-family:sans-serif;font-size:13px;padding:4px;">'
        f'<b>{p["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{p["model"]} · {p["dim"]}d</span></div>'
    )


def _set_empty_panel(i):
    headers[i].value = _empty_header(i)
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:80px 10px;">'
            'Ask a question to see<br>retrieved chunks + LLM answer</div>'
        ))


for i in range(len(EMBEDDING_PROVIDERS)):
    _set_empty_panel(i)

panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(EMBEDDING_PROVIDERS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(3, 1fr)", grid_gap="8px"),
)

summary_out = Output()

display(VBox([
    HBox([sample_dropdown]),
    question_box,
    HBox([ask_btn, clear_btn]),
    grid,
    summary_out,
]))

## 4. Wire it up

In [ ]:
import html as _html

import nest_asyncio
from IPython.display import clear_output

nest_asyncio.apply()

display(HTML('''
<style>
@keyframes cb_blink { 0%, 100% { opacity: 0.2; } 50% { opacity: 1; } }
.cb-dot { animation: cb_blink 1.2s infinite both; display:inline-block; }
.cb-dot:nth-child(2) { animation-delay: 0.2s; }
.cb-dot:nth-child(3) { animation-delay: 0.4s; }
</style>
'''))


def _render_loading(i):
    p = EMBEDDING_PROVIDERS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'<div><b>{p["label"]}</b> <span style="color:#888;font-size:11px;">{p["model"]} · {p["dim"]}d</span></div>'
        '<div><span style="background:#17a2b8;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">retrieving'
        '<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></span></div>'
        '</div>'
    )
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:80px 10px;">'
            'retrieving<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></div>'
        ))


def _render_header(i, retrieval_latency, answer_latency, status_color="#28a745", status_label="done ✓"):
    p = EMBEDDING_PROVIDERS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'<div><b>{p["label"]}</b> <span style="color:#888;font-size:11px;">{p["model"]} · {p["dim"]}d</span></div>'
        f'<div><span style="background:{status_color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">'
        f'{status_label}</span> '
        f'<span style="color:#666;font-size:11px;">'
        f'retrieve {retrieval_latency:.2f}s · answer {answer_latency:.2f}s</span></div>'
        '</div>'
    )


def _render_panel(i, retrieval, answer):
    panels[i].clear_output()
    _render_header(i, retrieval["latency"], answer["latency"])
    with panels[i]:
        chunks_html = ""
        for j, c in enumerate(retrieval["top"]):
            chunks_html += (
                f'<div style="display:flex;gap:8px;margin:4px 0;">'
                f'<div style="font-family:monospace;font-size:10px;color:#666;width:30px;">[{j+1}]</div>'
                f'<div style="font-family:monospace;font-size:10px;color:#666;width:50px;">{c["score"]:.3f}</div>'
                f'<div style="font-family:sans-serif;font-size:11px;flex:1;color:#333;">'
                f'{_html.escape(c["text"][:200])}{"…" if len(c["text"]) > 200 else ""}</div>'
                f'</div>'
            )
        ans_html = _html.escape(answer["answer"]).replace("\n", "<br>")
        display(HTML(
            f'<div style="font-family:sans-serif;font-size:11px;color:#666;text-transform:uppercase;'
            f'letter-spacing:0.5px;margin-bottom:4px;"><b>Top {TOP_K} chunks</b></div>'
            f'{chunks_html}'
            f'<div style="font-family:sans-serif;font-size:11px;color:#666;text-transform:uppercase;'
            f'letter-spacing:0.5px;margin-top:12px;margin-bottom:4px;"><b>Answer ({LLM_MODEL})</b></div>'
            f'<div style="background:#f8f9fa;padding:8px 10px;border-radius:4px;border-left:3px solid #007bff;'
            f'font-family:sans-serif;font-size:12px;line-height:1.5;">{ans_html}</div>'
        ))


def _render_error_panel(i, exc):
    p = EMBEDDING_PROVIDERS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'<div><b>{p["label"]}</b> <span style="color:#888;font-size:11px;">{p["model"]}</span></div>'
        '<div><span style="background:#dc3545;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">error ✗</span></div>'
        '</div>'
    )
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            f'<div style="color:#dc3545;font-family:monospace;font-size:11px;padding:8px;'
            f'background:#f8d7da;border-radius:4px;">{_html.escape(str(exc))[:400]}</div>'
        ))


async def _run_one(session, provider, question):
    retrieval = await retrieve(session, provider, question, CORPUS, TOP_K)
    answer = await answer_with_chunks(session, LLM_MODEL, question, retrieval["top"])
    return retrieval, answer


def _render_summary(all_results, question):
    # Did the providers agree on the top chunk?
    top_picks = []
    for (retrieval, _answer) in all_results:
        if retrieval["top"]:
            top_picks.append(retrieval["top"][0]["chunk_idx"])
    unique = len(set(top_picks))
    if unique == 1:
        agreement = '<span style="color:#28a745;">✓ All providers retrieved the same #1 chunk</span>'
    else:
        agreement = f'<span style="color:#fd7e14;">⚠ Providers disagree on the top chunk ({unique} different choices)</span>'

    union = set()
    for (retrieval, _answer) in all_results:
        for c in retrieval["top"]:
            union.add(c["chunk_idx"])
    with summary_out:
        clear_output()
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #007bff;font-family:sans-serif;font-size:13px;'
            f'display:flex;gap:18px;flex-wrap:wrap;">'
            f'{agreement}'
            f'<span style="color:#666;">📦 Union of retrieved chunks across providers: {len(union)} / {len(CORPUS)}</span>'
            f'</div>'
        ))


async def run_round(question):
    for i in range(len(EMBEDDING_PROVIDERS)):
        _render_loading(i)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[
            _run_one(session, EMBEDDING_PROVIDERS[i], question)
            for i in range(len(EMBEDDING_PROVIDERS))
        ], return_exceptions=True)
    ok_results = []
    for i, r in enumerate(results):
        if isinstance(r, Exception):
            _render_error_panel(i, r)
        else:
            _render_panel(i, r[0], r[1])
            ok_results.append(r)
    if ok_results:
        _render_summary(ok_results, question)


def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    asyncio.run(run_round(q))


def on_clear(_):
    for i in range(len(EMBEDDING_PROVIDERS)):
        _set_empty_panel(i)
    summary_out.clear_output()


ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

## 5. What to watch

Try the same query against different embedding providers:

- **"How much does Cinder cost?"** — every provider should pick chunk #3 (the pricing tier). If one picks chunk #12 (the Team-to-Enterprise upgrade), that's a retrieval miss.
- **"Is Cinder safe to use for healthcare data?"** — the right chunk is #9 (SOC2 + HIPAA), but providers can be tempted by chunk #6 (PINT benchmark, which uses similar safety vocabulary).
- **"Which frameworks does Cinder integrate with?"** — chunk #7 is the obvious answer. If a provider returns #4 (about the founders' previous companies — Stripe, Datadog), that's a classic dense-embedding hallucination of "frameworks".

When providers disagree, **the LLM gets different context → different answers** even with the same query and same answer model. That's the whole point of the demo: retrieval quality is upstream of everything.

## 6. Customize

**Swap the LLM.** `LLM_MODEL = "openai/gpt-4o"` — controls for retrieval, varies answer-generation.

**Compare LLMs *with* embedding providers fixed.** Pick one embedding model and 4 LLMs — same chunks, different answers. The simplest version of the [LLM Arena](llm_arena.ipynb), grounded.

**Bigger corpus.** Replace `CORPUS` with paragraphs from your own docs (split by header or fixed window). For >100 chunks you'd want to cache embeddings to disk:

```python
cache_path = pathlib.Path(f"embeds_{provider['label']}.json")
if cache_path.exists():
    chunk_vecs = json.loads(cache_path.read_text())
else:
    chunk_vecs = (await embed_batch(session, provider['model'], CORPUS))
    cache_path.write_text(json.dumps(chunk_vecs))
```

**Hybrid retrieval.** Combine cosine similarity with BM25 keyword matching for queries that need exact term recall (e.g. error codes, version numbers).